# 4. VAR-CLIP: Colab Text-to-Image Smoke Test

This notebook runs the official **VAR-CLIP-d16** text-to-image model on a Google Colab GPU.

```text
prompt -> CLIP ViT-L/14 text encoder -> 768-D text condition
       -> VAR-CLIP autoregressive token prediction -> VAR VAE decoder -> image grid
```

VAR-CLIP is different from our class-conditioned VAR-d20 experiments:

- VAR-d20 takes an ImageNet class ID.
- VAR-CLIP takes a CLIP text embedding.
- This notebook verifies text-conditioned generation only. It does not yet perform image-reference style transfer.

The official checkpoint is supplied from the authors' Google Drive. Keep the runtime on GPU; CPU inference is impractically slow.


In [ ]:
# Run this first. In Colab: Runtime -> Change runtime type -> T4 GPU (or better).
!nvidia-smi

import os
import subprocess
from pathlib import Path

assert os.path.exists('/usr/local/cuda') or os.environ.get('COLAB_GPU'), (
    'No GPU runtime detected. In Colab, select Runtime -> Change runtime type -> T4 GPU, then reconnect.'
)

if Path('/kaggle/working').exists():
    RUNTIME_ROOT = Path('/kaggle/working')
elif Path('/content').exists():
    RUNTIME_ROOT = Path('/content')
else:
    RUNTIME_ROOT = Path.cwd()

VAR_CLIP_REPO = 'https://github.com/daixiangzi/VAR-CLIP.git'
VAR_CLIP_DIR = RUNTIME_ROOT / 'VAR-CLIP'
OUTPUT_DIR = RUNTIME_ROOT / 'VAR_CLIP_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not VAR_CLIP_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', VAR_CLIP_REPO, str(VAR_CLIP_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(VAR_CLIP_DIR), 'fetch', 'origin', 'master'], check=True)
    subprocess.run(['git', '-C', str(VAR_CLIP_DIR), 'reset', '--hard', 'origin/master'], check=True)

os.chdir(VAR_CLIP_DIR)
print('VAR-CLIP source:', VAR_CLIP_DIR)
print('Outputs:', OUTPUT_DIR)
subprocess.run(['git', '-C', str(VAR_CLIP_DIR), 'log', '--oneline', '-1'], check=True)


In [ ]:
# Keep Colab's CUDA-enabled PyTorch. Do not install the repository's pinned torch~=2.1.0,
# because that can replace Colab's working GPU build.
!pip -q install gdown huggingface_hub einops typed-argument-parser pytz

import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
assert torch.cuda.is_available(), 'A CUDA GPU is required for this demo.'


## Download the Required Weights

VAR-CLIP needs three independent weights:

```text
VAR VAE:         FoundationVision/var on Hugging Face
VAR-CLIP-d16:    official author checkpoint on Google Drive
CLIP ViT-L/14:   downloaded automatically by the repository when first loaded
```

The author checkpoint is intentionally stored as `local_output/ar-ckpt-last.pth`, matching the official demo loader.


In [ ]:
from huggingface_hub import hf_hub_download
import gdown

PRETRAINED_DIR = VAR_CLIP_DIR / 'pretrained'
LOCAL_OUTPUT_DIR = VAR_CLIP_DIR / 'local_output'
PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

vae_path = Path(hf_hub_download(
    repo_id='FoundationVision/var',
    filename='vae_ch160v4096z32.pth',
    local_dir=PRETRAINED_DIR,
))

var_clip_path = LOCAL_OUTPUT_DIR / 'ar-ckpt-last.pth'
VAR_CLIP_CHECKPOINT_URL = 'https://drive.google.com/file/d/10gSxvaKaNKJcnqFhU7hQywU28w3nbgoV/view?usp=sharing'
if not var_clip_path.exists():
    gdown.download(url=VAR_CLIP_CHECKPOINT_URL, output=str(var_clip_path), fuzzy=True)

assert vae_path.exists(), vae_path
assert var_clip_path.exists() and var_clip_path.stat().st_size > 100_000_000, (
    'VAR-CLIP checkpoint download failed or is incomplete. Re-run this cell; Google Drive may temporarily rate-limit downloads.'
)
print('VAE:', vae_path, f'({vae_path.stat().st_size / 2**20:.0f} MiB)')
print('VAR-CLIP:', var_clip_path, f'({var_clip_path.stat().st_size / 2**20:.0f} MiB)')


## Load VAR-CLIP

The official model uses **CLIP ViT-L/14** to encode the prompt into a normalized 768-dimensional vector. That vector replaces the ImageNet class embedding used by ordinary VAR.

The first run can take longer because CLIP ViT-L/14 is downloaded into the runtime cache.


In [ ]:
import random
import numpy as np
import torch
import torchvision
import matplotlib.pyplot as plt
from PIL import Image

# Avoid unnecessary default initialization before loading large checkpoints.
setattr(torch.nn.Linear, 'reset_parameters', lambda self: None)
setattr(torch.nn.LayerNorm, 'reset_parameters', lambda self: None)

from clip_util import CLIPWrapper
from models.clip import clip_vit_l14
from tokenizer import tokenize
from models import build_vae_var

MODEL_DEPTH = 16
PATCH_NUMS = (1, 2, 3, 4, 5, 6, 8, 10, 13, 16)
device = 'cuda'

vae, var_clip = build_vae_var(
    V=4096,
    Cvae=32,
    ch=160,
    share_quant_resi=4,
    device=device,
    patch_nums=PATCH_NUMS,
    n_cond_embed=768,
    depth=MODEL_DEPTH,
    shared_aln=False,
)

clip_model = CLIPWrapper(clip_vit_l14(pretrained=True).to(device).eval(), normalize=True)

vae.load_state_dict(torch.load(vae_path, map_location='cpu'), strict=True)
checkpoint = torch.load(var_clip_path, map_location='cpu')
var_clip.load_state_dict(checkpoint['trainer']['var_wo_ddp'], strict=True)
vae.eval()
var_clip.eval()
for model in (vae, var_clip):
    for parameter in model.parameters():
        parameter.requires_grad_(False)

torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

print('VAR-CLIP-d16 and CLIP ViT-L/14 are ready.')


## Generate Images

Change `PROMPT`, then rerun this cell. Start with four images on a T4; increase `NUM_IMAGES` only after the smoke test succeeds.

`CFG` trades text adherence against diversity. `TOP_K` and `TOP_P` control token sampling. Keep the same `SEED` when comparing prompts or later interventions.


In [ ]:
PROMPT = 'a sailboat with a sail at sunset'  #@param {type:"string"}
NUM_IMAGES = 4                                 #@param {type:"integer"}
SEED = 0                                       #@param {type:"integer"}
CFG = 4.0                                      #@param {type:"number"}
TOP_K = 900                                    #@param {type:"integer"}
TOP_P = 0.95                                  #@param {type:"number"}
MORE_SMOOTH = False                            #@param {type:"boolean"}

assert 1 <= NUM_IMAGES <= 16, 'Use 1-16 images. Start with 4 on a T4 GPU.'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

text_tokens = tokenize([PROMPT] * NUM_IMAGES).to(device)
with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
    text_embeddings = clip_model.encode_text(text_tokens)
    images = var_clip.autoregressive_infer_cfg(
        B=NUM_IMAGES,
        label_B=text_embeddings,
        cfg=CFG,
        top_k=TOP_K,
        top_p=TOP_P,
        g_seed=SEED,
        more_smooth=MORE_SMOOTH,
    )

nrow = min(NUM_IMAGES, 4)
grid = torchvision.utils.make_grid(images, nrow=nrow, padding=2, pad_value=1.0)
grid_pil = Image.fromarray(
    grid.detach().float().clamp(0, 1).permute(1, 2, 0).mul(255).byte().cpu().numpy()
)

safe_prompt = ''.join(ch if ch.isalnum() else '_' for ch in PROMPT.lower()).strip('_')[:48]
out_path = OUTPUT_DIR / f'var_clip_d16_{safe_prompt}_seed{SEED}.png'
grid_pil.save(out_path)

plt.figure(figsize=(4 * nrow, 4 * ((NUM_IMAGES + nrow - 1) // nrow)))
plt.imshow(grid_pil)
plt.title(f'VAR-CLIP-d16 | "{PROMPT}" | seed={SEED}, cfg={CFG}')
plt.axis('off')
plt.show()
print('Saved:', out_path)


## Challenge Prompt Suite

This is a broader capability check, not a benchmark score. Each prompt is sampled independently, so it avoids putting every prompt in one large GPU batch.

The suite deliberately moves from familiar ImageNet-like concepts toward harder compositions:

```text
single subject          -> does the basic text condition work?
attribute binding       -> does the right attribute attach to the right object?
spatial relation        -> can the model organize two objects coherently?
material / art medium   -> can it apply a visual concept without collapsing the subject?
counting / typography   -> known difficult cases; useful for seeing the failure boundary
```

Use two samples per prompt first. This creates 24 images from 12 prompts and can take several minutes on a T4.


In [ ]:
from tqdm.auto import tqdm

RUN_CHALLENGE_SUITE = True  # Set False when you only want the single-prompt cell above.
SAMPLES_PER_PROMPT = 2      # Start with 2 on a T4. Raise to 4 after a successful run.
SUITE_CFG = 4.0
SUITE_TOP_K = 900
SUITE_TOP_P = 0.95
SUITE_MORE_SMOOTH = False
SUITE_SEED = 2026

CHALLENGE_PROMPTS = [
    # Familiar single-subject concepts: establishes the floor.
    'a golden retriever sitting in a field',
    'a red sports car on a mountain road',
    'a lighthouse on a rocky coast at sunset',

    # Attribute binding: one object with a specific appearance.
    'a small blue bird with yellow wings on a branch',
    'a red apple in a transparent glass bowl',
    'a white teddy bear wearing a green scarf',

    # Spatial / interaction relations: harder than a single noun.
    'a cat sitting beside a red bicycle',
    'a sailboat in front of a snow covered mountain',
    'a wooden chair under a large green tree',

    # Medium and lighting: visual style combined with semantic content.
    'a watercolor painting of a castle above a lake',
    'a cinematic night photograph of a city street in the rain',

    # Deliberately difficult: tests known weak points rather than expecting perfection.
    'three yellow birds flying over a purple flower field',
]

assert 1 <= SAMPLES_PER_PROMPT <= 8, 'Use 1-8 samples per prompt on a Colab T4.'

if RUN_CHALLENGE_SUITE:
    suite_records = []
    all_images_cpu = []

    for prompt_id, prompt in enumerate(tqdm(CHALLENGE_PROMPTS, desc='VAR-CLIP challenge prompts')):
        prompt_seed = SUITE_SEED + prompt_id
        random.seed(prompt_seed)
        np.random.seed(prompt_seed)
        torch.manual_seed(prompt_seed)
        torch.cuda.manual_seed_all(prompt_seed)

        tokens = tokenize([prompt] * SAMPLES_PER_PROMPT).to(device)
        with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
            embeddings = clip_model.encode_text(tokens)
            generated = var_clip.autoregressive_infer_cfg(
                B=SAMPLES_PER_PROMPT,
                label_B=embeddings,
                cfg=SUITE_CFG,
                top_k=SUITE_TOP_K,
                top_p=SUITE_TOP_P,
                g_seed=prompt_seed,
                more_smooth=SUITE_MORE_SMOOTH,
            )

        # Move finished samples off GPU before processing the next prompt.
        generated_cpu = generated.detach().float().cpu().clamp(0, 1)
        prompt_slug = ''.join(ch if ch.isalnum() else '_' for ch in prompt.lower()).strip('_')[:48]
        prompt_path = OUTPUT_DIR / f'challenge_{prompt_id:02d}_{prompt_slug}_seed{prompt_seed}.png'
        prompt_grid = torchvision.utils.make_grid(generated_cpu, nrow=SAMPLES_PER_PROMPT, padding=2, pad_value=1.0)
        Image.fromarray(prompt_grid.permute(1, 2, 0).mul(255).byte().numpy()).save(prompt_path)

        for sample_id, image in enumerate(generated_cpu):
            all_images_cpu.append(image.unsqueeze(0))
            suite_records.append({
                'prompt_id': prompt_id,
                'sample_id': sample_id,
                'seed': prompt_seed,
                'prompt': prompt,
                'output_path': str(prompt_path),
            })

        del generated, generated_cpu, tokens, embeddings
        torch.cuda.empty_cache()

    # One contact sheet: rows are prompts, columns are samples.
    contact = torchvision.utils.make_grid(
        torch.cat(all_images_cpu, dim=0),
        nrow=SAMPLES_PER_PROMPT,
        padding=4,
        pad_value=1.0,
    )
    contact_path = OUTPUT_DIR / f'var_clip_d16_challenge_suite_seed{SUITE_SEED}.png'
    Image.fromarray(contact.permute(1, 2, 0).mul(255).byte().numpy()).save(contact_path)

    suite_df = pd.DataFrame(suite_records)
    suite_csv_path = OUTPUT_DIR / f'var_clip_d16_challenge_suite_seed{SUITE_SEED}.csv'
    suite_df.to_csv(suite_csv_path, index=False)

    rows = len(CHALLENGE_PROMPTS)
    plt.figure(figsize=(4 * SAMPLES_PER_PROMPT, 4 * rows))
    plt.imshow(Image.open(contact_path))
    plt.axis('off')
    plt.title(f'VAR-CLIP-d16 challenge suite | {SAMPLES_PER_PROMPT} samples per prompt')
    plt.show()

    display(suite_df[['prompt_id', 'sample_id', 'seed', 'prompt']])
    print('Contact sheet:', contact_path)
    print('Prompt manifest:', suite_csv_path)


## What This Verifies

```text
Successful result:
    CLIP tokenization, CLIP text encoding, VAR-CLIP checkpoint loading, and autoregressive
    image decoding all work together on the Colab GPU.

Important boundary:
    this is text-conditioned VAR-CLIP, not content-image-conditioned style transfer.
    It is useful as a text-conditioned VAR baseline and as a possible future source of semantic
    text control, but it does not replace Notebook 3.2's reference-image PFB + SAC experiment.
```
